<a href="https://colab.research.google.com/github/russianoracle/punct-nlu-colab-training/blob/main/PunctNLU_Colab_Training_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PunctNLU CUDA Training on Colab
Training Russian punctuation restoration model on GPU

**Model:** rubert-tiny2 (29.2M params) → 12-class punctuation + capitalization

**Corpus:** 1.194M Russian sentences from Tatoeba

**Hardware:** Colab GPU (T4/A100)

## Step 1: Setup & Install Dependencies

In [17]:
!pip install -q torch transformers numpy pandas scikit-learn tqdm

In [18]:
import logging
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hugging Face Authentication ────────────────────────────────────────────────
try:
    from google.colab import userdata
    import os

    # Check both common variations
    token = None
    for key in ['HF_TOKEN', 'hf_token']:
        try:
            token = userdata.get(key)
            if token:
                os.environ['HF_TOKEN'] = token
                print(f"✓ {key} found and loaded from Colab Secrets")
                break
        except:
            continue

    if not token:
        print("⚠️  HF_TOKEN not detected in Secrets.")
        print("   Please ensure you added it under 🔑 Secrets AND enabled 'Notebook access'.")
except Exception as e:
    print(f"Could not access Colab Secrets: {e}")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.6 GB
✓ HF_TOKEN found and loaded from Colab Secrets


In [19]:
from google.colab import drive

# Optional: Mount Drive only if you need to save results there
# drive.mount('/content/drive', force_remount=True)
print("✓ Colab environment ready")

✓ Colab environment ready


In [20]:
from google.colab import drive, files
from pathlib import Path
import shutil
import os

print("=" * 80)
print("CORPUS DETECTION")
print("=" * 80)

# Check if corpus already exists in /content/
CORPUS_PATH = Path("/content/sentences.csv")
print(f"1️⃣  Checking /content/sentences.csv... {CORPUS_PATH.exists()}")

if CORPUS_PATH.exists():
    size_mb = CORPUS_PATH.stat().st_size / 1e6
    print(f"   ✓ Found: {size_mb:.0f} MB")
else:
    # Mount Google Drive as fallback
    print("2️⃣  /content not found, mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    # Create directory for corpus
    corpus_dir = Path("/content/drive/MyDrive/PunctNLU")
    corpus_dir.mkdir(parents=True, exist_ok=True)
    print(f"   Created: {corpus_dir}")

    CORPUS_PATH = corpus_dir / "sentences.csv"
    print(f"3️⃣  Checking Google Drive... {CORPUS_PATH.exists()}")

    # Check if corpus exists in Drive
    if CORPUS_PATH.exists():
        size_mb = CORPUS_PATH.stat().st_size / 1e6
        print(f"   ✓ Found: {size_mb:.0f} MB")
    else:
        print("4️⃣  Not in Drive either, prompting for upload...")
        print("   Click 'Choose Files' below and select sentences.csv")

        uploaded = files.upload()
        if 'sentences.csv' in uploaded:
            shutil.move('./sentences.csv', str(CORPUS_PATH))
            size_mb = CORPUS_PATH.stat().st_size / 1e6
            print(f"   ✓ Uploaded: {size_mb:.0f} MB")

print(f"\nFinal CORPUS_PATH: {CORPUS_PATH}")
print(f"Type: {type(CORPUS_PATH)}")
print(f"Exists: {CORPUS_PATH.exists()}")
print("=" * 80)

CORPUS DETECTION
1️⃣  Checking /content/sentences.csv... True
   ✓ Found: 458 MB

Final CORPUS_PATH: /content/sentences.csv
Type: <class 'pathlib.PosixPath'>
Exists: True


### 👆 Click above cell to mount Drive and upload corpus
1. Run cell above
2. If corpus not found, click "Choose Files" button
3. Select `sentences.csv` from your computer
4. Colab will upload it to Google Drive automatically (first run only ~5-10 min)
5. Run remaining cells for training!
✅ Notebook handles everything else

## Step 3: Configuration (CUDA-Optimized)

In [21]:
# ── Config ─────────────────────────────────────────────────────────────────

MODEL_ID      = "cointegrated/rubert-tiny2"
SEQ_LEN       = 64
NUM_LABELS    = 12
BATCH_SIZE    = 128  # Higher on GPU (was 16 on M2)
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 1e-5
NUM_WORKERS   = 2
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR    = Path("./checkpoints/punct_nlu")

PUNCT_CHARS   = {",": 1, ".": 2, "!": 2, "…": 2, ";": 2, "?": 3}

# CORPUS_PATH is already set in cell 6 (mounted from Google Drive)
# Ensure it's a Path object
if isinstance(CORPUS_PATH, str):
    CORPUS_PATH = Path(CORPUS_PATH)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 80)
print("CONFIGURATION SUMMARY")
print("=" * 80)
print(f"Device:        {DEVICE}")
print(f"Batch Size:    {BATCH_SIZE}")
print(f"Model ID:      {MODEL_ID}")
print(f"Corpus Path:   {CORPUS_PATH}")
print(f"Corpus Exists: {CORPUS_PATH.exists()}")
print(f"Output Dir:    {OUTPUT_DIR}")
print(f"Epochs:        {NUM_EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print("=" * 80)


CONFIGURATION SUMMARY
Device:        cuda
Batch Size:    128
Model ID:      cointegrated/rubert-tiny2
Corpus Path:   /content/sentences.csv
Corpus Exists: True
Output Dir:    checkpoints/punct_nlu
Epochs:        3
Learning Rate: 0.0001


## Step 4: Data Loading & Processing

In [22]:
# ── Label extraction ───────────────────────────────────────────────────────
from typing import Optional

def cap_mode_vec(words: np.ndarray) -> np.ndarray:
    """Vectorized capitalization detection (faster than list comp)"""
    modes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        clean = re.sub(r"[^\w]", "", word, flags=re.UNICODE)
        if clean:
            alpha = [c for c in clean if c.isalpha()]
            if alpha:
                if all(c.isupper() for c in alpha):
                    modes[i] = 2
                elif alpha[0].isupper():
                    modes[i] = 1
    return modes


def punct_class_vec(words: np.ndarray) -> np.ndarray:
    """Vectorized punctuation detection (faster than list comp)"""
    classes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        if word:
            last = word[-1]
            classes[i] = PUNCT_CHARS.get(last, 0)
    return classes


def word_labels_vec(words: list[str]) -> np.ndarray:
    """Vectorized label creation from word list"""
    words_arr = np.array(words, dtype=object)
    punct_classes = punct_class_vec(words_arr)
    cap_modes = cap_mode_vec(words_arr)
    return punct_classes * 3 + cap_modes


def ortho_to_norm(word: str) -> str:
    return re.sub(r"[^\w]", "", word, flags=re.UNICODE).lower()

In [23]:
# ── Data loader ────────────────────────────────────────────────────────────

def iter_tatoeba(path: Path) -> Iterator[tuple[list[str], list[int]]]:
    """Read Tatoeba corpus without limits"""
    count = 0
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 2)
            if len(parts) != 3 or parts[1] != "rus":
                continue

            sentence = parts[2].strip()
            if len(sentence) < 5:
                continue

            words = sentence.split()
            if len(words) < 2:
                continue

            labels = word_labels_vec(words).tolist()
            norms = [ortho_to_norm(w) for w in words]

            if not any(norms):
                continue

            yield norms, labels
            count += 1
            if count % 100000 == 0:
                log.info(f"  Loaded {count} sentences...")


@dataclass
class EncodedSample:
    input_ids: torch.Tensor      # [SEQ_LEN]
    attention_mask: torch.Tensor # [SEQ_LEN]
    labels: torch.Tensor         # [SEQ_LEN]


def encode(norm_words: list[str], word_labels: list[int], tokenizer) -> Optional[EncodedSample]:
    """Encode sentence with proper word-start alignment"""
    ids = np.zeros(SEQ_LEN, dtype=np.int32)
    mask = np.zeros(SEQ_LEN, dtype=np.int32)
    labels = np.full(SEQ_LEN, -100, dtype=np.int32)

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    ids[0] = cls_id
    mask[0] = 1
    pos = 1

    for word, lbl in zip(norm_words, word_labels):
        if pos >= SEQ_LEN - 1:
            break

        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if not sub_ids:
            sub_ids = [tokenizer.unk_token_id]

        if pos + len(sub_ids) >= SEQ_LEN:
            break

        ids[pos] = sub_ids[0]
        mask[pos] = 1
        labels[pos] = lbl
        pos += 1

        for sub_id in sub_ids[1:]:
            if pos >= SEQ_LEN:
                break
            ids[pos] = sub_id
            mask[pos] = 1
            labels[pos] = -100
            pos += 1

    if pos < SEQ_LEN:
        ids[pos] = sep_id
        mask[pos] = 1
        pos += 1

    return EncodedSample(
        input_ids=torch.from_numpy(ids).long(),
        attention_mask=torch.from_numpy(mask).long(),
        labels=torch.from_numpy(labels).long(),
    )


class PunctDataset(Dataset):
    def __init__(self, samples: list[EncodedSample]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.labels,
        }

In [24]:
import pickle

def save_encoded_samples(samples, filepath="/content/encoded_samples.pkl"):
    """Saves the list of EncodedSample objects to disk."""
    with open(filepath, 'wb') as f:
        pickle.dump(samples, f)
    print(f"✓ Successfully saved {len(samples)} samples to {filepath}")

def load_encoded_samples(filepath="/content/encoded_samples.pkl"):
    """Loads the list of EncodedSample objects from disk."""
    if Path(filepath).exists():
        with open(filepath, 'rb') as f:
            samples = pickle.load(f)
        print(f"✓ Loaded {len(samples)} samples from {filepath}")
        return samples
    else:
        print(f"✗ No saved samples found at {filepath}")
        return None

## Step 5: Model Definition

In [25]:
# ── Model ──────────────────────────────────────────────────────────────────

class PunctNLUModel(nn.Module):
    """Two-head model: punct restoration + actionable classification"""

    def __init__(self, model_id: str, num_labels: int = 12):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_id)
        self.dropout = nn.Dropout(0.1)

        # Head A1: Punctuation restoration (12 classes)
        self.punct_head = nn.Linear(self.bert.config.hidden_size, num_labels)

        # Head A2: Actionable classification (2 classes)
        self.classify_head = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # [B, L, H]
        hidden = self.dropout(hidden)

        punct_logits = self.punct_head(hidden)      # [B, L, 12]
        classify_logits = self.classify_head(hidden) # [B, L, 2]

        return punct_logits, classify_logits

## Step 6: Training Loop

In [26]:
def train_epoch(model, loader, optimizer, scheduler, device, scaler=None):
    """Train for one epoch with Mixed Precision Training"""
    model.train()
    total_loss = 0.0
    batch_losses = []

    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                punct_logits, classify_logits = model(input_ids, attention_mask)
                loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
                binary_labels = (labels > 0).long()
                loss_classify = ce_loss_fn(classify_logits.view(-1, 2), binary_labels.view(-1))
                loss = loss_punct + 0.5 * loss_classify
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            punct_logits, classify_logits = model(input_ids, attention_mask)
            loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
            binary_labels = (labels > 0).long()
            loss_classify = ce_loss_fn(classify_logits.view(-1, 2), binary_labels.view(-1))
            loss = loss_punct + 0.5 * loss_classify
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        loss_val = loss.item()
        total_loss += loss_val
        batch_losses.append(loss_val)

        if (batch_idx + 1) % 100 == 0:
            avg_loss = np.mean(batch_losses[-100:])
            lr = optimizer.param_groups[0]['lr']
            gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            # Using print(..., flush=True) for immediate visibility in Colab
            print(f"  Batch {batch_idx + 1:4d}/{len(loader)} | loss: {loss_val:.4f} | avg_loss: {avg_loss:.4f} | GPU: {gpu_mem:.2f}GB | LR: {lr:.2e}", flush=True)

    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    correct = torch.tensor(0, device=device)
    total = torch.tensor(0, device=device)
    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            punct_logits, _ = model(input_ids, attention_mask)
            loss = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
            total_loss += loss.item()
            pred = torch.argmax(punct_logits, dim=-1)
            mask = labels >= 0
            correct += (pred[mask] == labels[mask]).sum()
            total += mask.sum()
    return total_loss / len(loader), (correct / max(total, 1)).item()

## Step 7: Main Training Function

In [27]:
def main():
    print("\n" + "="*50, flush=True)
    print("🚀 STARTING TRAINING PROCESS", flush=True)
    print("="*50, flush=True)

    global DEVICE
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        print(f"✓ Using Device: {torch.cuda.get_device_name(0)}", flush=True)

    # ── 1. Load tokenizer ───────────────────────────────────────────────────
    print(f"🔤 Loading tokenizer: {MODEL_ID}...", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    # ── 2. Load or Encode samples ───────────────────────────────────────────
    CACHE_FILE = "/content/encoded_samples.pkl"
    encoded_samples = load_encoded_samples(CACHE_FILE)

    if encoded_samples is None:
        print(f"📂 Loading corpus from {CORPUS_PATH}...", flush=True)
        corpus_data = list(iter_tatoeba(CORPUS_PATH))
        print(f"✓ Loaded {len(corpus_data):,} sentences", flush=True)

        print(f"⚙️  Encoding samples...", flush=True)
        encoded_samples = []
        for idx, (norm_words, labels) in enumerate(corpus_data):
            sample = encode(norm_words, labels, tokenizer)
            if sample: encoded_samples.append(sample)
            if (idx + 1) % 100000 == 0:
                print(f"  Progress: {(idx+1)/len(corpus_data)*100:.1f}% ({idx+1:,} sentences)", flush=True)

        # Save for next time
        save_encoded_samples(encoded_samples, CACHE_FILE)
    else:
        print(f"⚡ Skipping encoding phase, using cached samples.", flush=True)

    # ── 3. Split train/val ──────────────────────────────────────────────────
    split = int(0.95 * len(encoded_samples))
    train_dataset = PunctDataset(encoded_samples[:split])
    val_dataset = PunctDataset(encoded_samples[split:])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # ── 4. Initialize model ─────────────────────────────────────────────────
    print(f"🧠 Initializing model on {DEVICE}...", flush=True)
    model = PunctNLUModel(MODEL_ID).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=len(train_loader)*NUM_EPOCHS)
    scaler = torch.cuda.amp.GradScaler() if DEVICE == "cuda" else None

    # ── 5. Training loop ────────────────────────────────────────────────────
    print("\n" + "="*50, flush=True)
    print("STARTING EPOCHS", flush=True)
    print("="*50, flush=True)

    best_val_loss = float("inf")
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n▶ Epoch {epoch}/{NUM_EPOCHS}", flush=True)
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, DEVICE, scaler)
        val_loss, val_acc = eval_epoch(model, val_loader, DEVICE)

        print(f"  Done in {time.time()-t0:.0f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}", flush=True)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / "best.pt")
            print(f"  ⭐ New Best Model Saved!", flush=True)

    print("\n" + "="*50, flush=True)
    print("✅ TRAINING COMPLETE", flush=True)
    print("="*50, flush=True)
    return True

## Step 8: RUN TRAINING

In [ ]:
# ── GPU Recovery & Diagnostic ──────────────────────────────────────────────────

print("\n" + "=" * 80)
print("GPU RECOVERY & DIAGNOSTIC")
print("=" * 80)

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")
print(f"torch.cuda.get_device_name(0): {torch.cuda.get_device_name(0)}")
print(f"torch.version.cuda: {torch.version.cuda}")

# Clear GPU cache
print("\n🔄 Clearing GPU cache...")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Try to allocate a small tensor on GPU
print("Testing GPU allocation...")
try:
    test_tensor = torch.zeros(1, device="cuda")
    print(f"✓ GPU allocation successful")
    print(f"  Test tensor device: {test_tensor.device}")
    del test_tensor
    torch.cuda.empty_cache()
except Exception as e:
    print(f"✗ GPU allocation failed: {e}")
    print("\n💡 SOLUTIONS:")
    print("  1. Runtime → Restart runtime (clears GPU)")
    print("  2. Or reduce BATCH_SIZE in config (Cell 9)")
    print("  3. Or use CPU by setting DEVICE='cpu'")
    raise

# Check model will be on GPU
print(f"\nConfigured device: {DEVICE}")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("=" * 80 + "\n")

print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)
print(f"Corpus path: {CORPUS_PATH}")
print(f"Corpus exists: {CORPUS_PATH.exists()}")
print(f"Device: {DEVICE}")
print("=" * 80 + "\n")

success = main()
if success:
    print("\n" + "=" * 80)
    print("✓ Training completed successfully!")
    print("=" * 80)
else:
    print("\n" + "=" * 80)
    print("✗ Training failed. Check logs above.")
    print("=" * 80)


GPU RECOVERY & DIAGNOSTIC
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
torch.cuda.get_device_name(0): Tesla T4
torch.version.cuda: 12.8

🔄 Clearing GPU cache...
Testing GPU allocation...
✓ GPU allocation successful
  Test tensor device: cuda:0

Configured device: cuda
GPU memory available: 15.6 GB


STARTING TRAINING
Corpus path: /content/sentences.csv
Corpus exists: True
Device: cuda


🚀 STARTING TRAINING PROCESS
✓ Using Device: Tesla T4
📂 Loading corpus from /content/sentences.csv...
✓ Loaded 761,723 sentences
🔤 Loading tokenizer: cointegrated/rubert-tiny2...
⚙️  Encoding samples...
  Progress: 13.1% (100,000 sentences)
  Progress: 26.3% (200,000 sentences)
  Progress: 39.4% (300,000 sentences)
  Progress: 52.5% (400,000 sentences)
  Progress: 65.6% (500,000 sentences)
  Progress: 78.8% (600,000 sentences)
  Progress: 91.9% (700,000 sentences)
🧠 Initializing model on cuda...


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STARTING EPOCHS

▶ Epoch 1/3


/tmp/ipykernel_70398/3370382600.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if DEVICE == "cuda" else None
/tmp/ipykernel_70398/1092008818.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_70398/1092008818.py:38: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


...

  Batch 5100/5654 | loss: 0.0847 | avg_loss: 0.0758 | GPU: 0.49GB | LR: 4.29e-05
  Batch 5200/5654 | loss: 0.0726 | avg_loss: 0.0777 | GPU: 0.49GB | LR: 4.24e-05
  Batch 5300/5654 | loss: 0.0669 | avg_loss: 0.0776 | GPU: 0.49GB | LR: 4.19e-05
  Batch 5400/5654 | loss: 0.0889 | avg_loss: 0.0753 | GPU: 0.49GB | LR: 4.13e-05
  Batch 5500/5654 | loss: 0.0755 | avg_loss: 0.0750 | GPU: 0.49GB | LR: 4.08e-05
  Batch 5600/5654 | loss: 0.0822 | avg_loss: 0.0736 | GPU: 0.49GB | LR: 4.03e-05


/tmp/ipykernel_70398/1092008818.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Done in 285s | Train Loss: 0.0795 | Val Loss: 0.0716 | Val Acc: 0.977
  ⭐ New Best Model Saved!

▶ Epoch 3/3
  Batch  100/5654 | loss: 0.0705 | avg_loss: 0.0651 | GPU: 0.49GB | LR: 3.95e-05


/tmp/ipykernel_70398/1092008818.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Batch  200/5654 | loss: 0.0713 | avg_loss: 0.0646 | GPU: 0.49GB | LR: 3.89e-05
  Batch  300/5654 | loss: 0.0970 | avg_loss: 0.0659 | GPU: 0.49GB | LR: 3.84e-05
  Batch  400/5654 | loss: 0.0436 | avg_loss: 0.0656 | GPU: 0.49GB | LR: 3.79e-05
  Batch  500/5654 | loss: 0.0916 | avg_loss: 0.0640 | GPU: 0.49GB | LR: 3.73e-05
  Batch  600/5654 | loss: 0.0780 | avg_loss: 0.0647 | GPU: 0.49GB | LR: 3.68e-05
  Batch  700/5654 | loss: 0.0567 | avg_loss: 0.0678 | GPU: 0.49GB | LR: 3.63e-05
  Batch  800/5654 | loss: 0.0514 | avg_loss: 0.0633 | GPU: 0.49GB | LR: 3.58e-05
  Batch  900/5654 | loss: 0.0339 | avg_loss: 0.0644 | GPU: 0.49GB | LR: 3.52e-05
  Batch 1000/5654 | loss: 0.0836 | avg_loss: 0.0661 | GPU: 0.49GB | LR: 3.47e-05
  Batch 1100/5654 | loss: 0.0422 | avg_loss: 0.0641 | GPU: 0.49GB | LR: 3.42e-05
  Batch 1200/5654 | loss: 0.0572 | avg_loss: 0.0675 | GPU: 0.49GB | LR: 3.36e-05
  Batch 1300/5654 | loss: 0.0602 | avg_loss: 0.0639 | GPU: 0.49GB | LR: 3.31e-05
  Batch 1400/5654 | loss: 0.

In [ ]:
# Define the persistent path in your Google Drive
DRIVE_CACHE_PATH = "/content/drive/MyDrive/PunctNLU/encoded_samples.pkl"

if 'encoded_samples' in locals() and encoded_samples:
    print(f"💾 Saving current progress to Drive...")
    save_encoded_samples(encoded_samples, DRIVE_CACHE_PATH)
else:
    print("⚠️ Variable 'encoded_samples' not found in memory. Please ensure the encoding step finished successfully.")

In [ ]:
import cProfile
import pstats
import io

# We'll profile the main function to see the distribution of time
pr = cProfile.Profile()
pr.enable()

# Running a smaller subset or the full main to gather stats
# Note: If your dataset is huge, you might want to wrap just the encoding loop
# inside main() with these commands instead.
success = main()

pr.disable()
s = io.StringIO()
sortby = 'cumulative'  # Can also use 'tottime' to see exclusive time
ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
ps.print_stats(20)  # Print the top 20 time-consuming functions
print(s.getvalue())

## Step 9: Download Results

In [ ]:
from google.colab import files
from pathlib import Path

# Check if training completed
checkpoint_dir = Path("./checkpoints/punct_nlu")
best_model = checkpoint_dir / "best.pt"
final_model = checkpoint_dir / "punct_nlu.pt"

if best_model.exists():
    print(f"Downloading trained model: {best_model.stat().st_size / 1e6:.0f} MB")
    files.download(str(best_model))
    print("✓ Downloaded: best.pt")
elif final_model.exists():
    print(f"Downloading final checkpoint: {final_model.stat().st_size / 1e6:.0f} MB")
    files.download(str(final_model))
    print("✓ Downloaded: punct_nlu.pt")
else:
    print("✗ Training output not found!")
    print(f"Checked: {checkpoint_dir}")
    print("
Did training complete successfully?")
    print("Check the training log above for errors.")